<a href="https://colab.research.google.com/github/subiksha0515/Recommendation_of_RAG/blob/main/Two_Embedding_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Embedding Models and Trade-offs — A Simple Experiment

This notebook compares:

- 🧠 An open-source embedding model  
- ☁️ An API-based embedding model  

Using the same text chunks and queries, we analyze:

- Embedding dimensions
- Semantic retrieval quality
- Paraphrase robustness
- Performance and latency trade-offs

This notebook is **fully customizable** — you can change only the chunks and queries while keeping the experiment flow intact.


🟦 PART A — Embedding Model Setup
🧩 Install Required Libraries (Code Cell)

In [ ]:
!pip install sentence-transformers numpy openai


📥 Import Required Libraries (Code Cell)

In [ ]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI




🧠 Initialize Embedding Models (Code Cell)
This cell loads:

- An open-source embedding model  
- An API-based embedding model  

It also stores:

- The model names  
- The model objects  

These models will be used for all later experiments.

In [ ]:
# Open-source embedding model
open_source_model_name = "multi-qa-MiniLM-L6-cos-v1"
open_source_model = SentenceTransformer(open_source_model_name)

# API-based embedding model
api_key = "YOUR_API_KEY"       # Replace with your valid API key
base_url = "https://apidev.navigatelabsai.com/"     # Replace with your valid base URL if needed

client = OpenAI(api_key=api_key, base_url=base_url)
api_model_name = "text-embedding-3-small"

print(f"Type of open_source_model: {type(open_source_model)}")
print(f"Type of API client: {type(client)}")


Type of open_source_model: <class 'sentence_transformers.SentenceTransformer.SentenceTransformer'>
Type of API client: <class 'openai.OpenAI'>


🟦 PART B — Define Your Custom Data
📄 Define Text Chunks (CUSTOMIZE HERE) ✏️ (Code Cell)

In [ ]:
chunks = [
    """
    Customers can return products within 30 days of purchase.
    Items must be unused, in original condition, and include all packaging.
    Returns are initiated through the customer account portal.
    Once approved, a return shipping label is provided.
    """,

    """
    Refunds are processed after the returned item is received and inspected.
    The inspection ensures the product meets return conditions.
    Approved refunds are issued within five business days.
    The time taken to reflect in the bank depends on the payment provider.
    """,

    """
    Free shipping is available for orders above 100 dollars.
    This offer applies only to standard delivery.
    Expedited shipping options are available for an additional fee.
    Free shipping may not apply to oversized or heavy items.
    """,

    """
    Customer support is available 24 hours a day.
    Support can be reached via live chat, email, or phone.
    Live chat is recommended for quick issue resolution.
    Email support is suitable for detailed or documented issues.
    """,

    """
    Discount coupons must be applied during checkout.
    Coupons have expiration dates and usage conditions.
    They cannot be combined with other promotions.
    Expired or invalid coupons cannot be reused.
    """
]


🧠 Generate Embeddings — Open-Source Model (Code Cell)

In [ ]:
start_time = time.time()
open_source_embeddings = open_source_model.encode(chunks)
open_source_time_all = time.time() - start_time

print("Open-source model:", open_source_model_name)
print("Embedding dimension:", open_source_embeddings.shape[1])
print("Number of chunks embedded:", open_source_embeddings.shape[0])
print(f"Embedding shape: {open_source_embeddings.shape[0]} × {open_source_embeddings.shape[1]}")


Open-source model: multi-qa-MiniLM-L6-cos-v1
Embedding dimension: 384
Number of chunks embedded: 5
Embedding shape: 5 × 384


☁️ Generate Embeddings — API Model (Code Cell)

In [ ]:
start_time = time.time()
open_source_embeddings = open_source_model.encode(chunks)
open_source_time_all = time.time() - start_time

print("Open-source model:", open_source_model_name)
print("Embedding dimension:", open_source_embeddings.shape[1])
print("Number of chunks embedded:", open_source_embeddings.shape[0])


Open-source model: multi-qa-MiniLM-L6-cos-v1
Embedding dimension: 384
Number of chunks embedded: 5



📏 Compare Embedding Dimensions (Code Cell)
python
Copy code


In [ ]:
start_time = time.time()

response = client.embeddings.create(
    model=api_model_name,
    input=chunks
)

api_embeddings = np.array([item.embedding for item in response.data])
api_time_all = time.time() - start_time

print("API model:", api_model_name)
print("Embedding dimension:", api_embeddings.shape[1])
print("Number of chunks embedded:", api_embeddings.shape[0])


API model: text-embedding-3-small
Embedding dimension: 1536
Number of chunks embedded: 5



🟦 PART C — Semantic Retrieval
❓ Define User Query (CUSTOMIZE HERE) ✏️ (Code Cell)
python
Copy code


In [ ]:
query = "How long does it take to process my refund?"



📐 Cosine Similarity Function (Code Cell)
python
Copy code


In [ ]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))



🔍 Retrieval — Open-Source Model (Code Cell)
python
Copy code


In [ ]:
query_emb_open = open_source_model.encode(query)

open_source_scores = [
    cosine_similarity(query_emb_open, emb)
    for emb in open_source_embeddings
]

open_source_ranked = sorted(
    enumerate(open_source_scores),
    key=lambda x: x[1],
    reverse=True
)

print("🔍 Open-Source Model Results")
for idx, score in open_source_ranked:
    print(f"Chunk {idx+1} | Score: {score:.4f}")


🔍 Open-Source Model Results
Chunk 2 | Score: 0.7335
Chunk 1 | Score: 0.4265
Chunk 4 | Score: 0.2199
Chunk 3 | Score: 0.0669
Chunk 5 | Score: -0.0292


☁️ Retrieval — API Model (Code Cell)

In [ ]:
response = client.embeddings.create(
    model=api_model_name,
    input=[query]
)

query_emb_api = np.array(response.data[0].embedding)

api_scores = [
    cosine_similarity(query_emb_api, emb)
    for emb in api_embeddings
]

api_ranked = sorted(
    enumerate(api_scores),
    key=lambda x: x[1],
    reverse=True
)

print("🔍 API Model Results")
for idx, score in api_ranked:
    print(f"Chunk {idx+1} | Score: {score:.4f}")


🔍 API Model Results
Chunk 2 | Score: 0.5947
Chunk 1 | Score: 0.3236
Chunk 4 | Score: 0.1696
Chunk 5 | Score: 0.1051
Chunk 3 | Score: 0.0933


🟦 PART D — Paraphrase Robustness
🔁 Define Paraphrased Query (CUSTOMIZE HERE) ✏️ (Code Cell)

In [ ]:
paraphrased_query = "When will my refund be completed after returning an item?"


🧠 Paraphrase Search — Open-Source (Code Cell)

In [ ]:
query_emb_open_para = open_source_model.encode(paraphrased_query)

open_source_scores_para = [
    cosine_similarity(query_emb_open_para, emb)
    for emb in open_source_embeddings
]

open_source_ranked_para = sorted(
    enumerate(open_source_scores_para),
    key=lambda x: x[1],
    reverse=True
)

print("🔁 Open-Source Model (Paraphrased Query)")
for idx, score in open_source_ranked_para:
    print(f"Chunk {idx+1} | Score: {score:.4f}")


🔁 Open-Source Model (Paraphrased Query)
Chunk 2 | Score: 0.6577
Chunk 1 | Score: 0.5625
Chunk 5 | Score: 0.1929
Chunk 4 | Score: 0.1843
Chunk 3 | Score: 0.0872


☁️ Paraphrase Search — API Model (Code Cell)

In [ ]:
response = client.embeddings.create(
    model=api_model_name,
    input=[paraphrased_query]
)

query_emb_api_para = np.array(response.data[0].embedding)

api_scores_para = [
    cosine_similarity(query_emb_api_para, emb)
    for emb in api_embeddings
]

api_ranked_para = sorted(
    enumerate(api_scores_para),
    key=lambda x: x[1],
    reverse=True
)

print("🔁 API Model (Paraphrased Query)")
for idx, score in api_ranked_para:
    print(f"Chunk {idx+1} | Score: {score:.4f}")


🔁 API Model (Paraphrased Query)
Chunk 2 | Score: 0.6181
Chunk 1 | Score: 0.4663
Chunk 5 | Score: 0.1724
Chunk 3 | Score: 0.1715
Chunk 4 | Score: 0.1272


🟦 PART E — Performance Comparison
⏱ Single Chunk Latency (Code Cell)

In [ ]:
single_chunk = [chunks[0]]

# Open-source
start_time = time.time()
_ = open_source_model.encode(single_chunk)
open_source_time_single = time.time() - start_time

# API
start_time = time.time()
_ = client.embeddings.create(model=api_model_name, input=single_chunk)
api_time_single = time.time() - start_time

print("⏱ Single Chunk Embedding Time")
print("Open-source model:", open_source_time_single, "seconds")
print("API model:", api_time_single, "seconds")


⏱ Single Chunk Embedding Time
Open-source model: 0.031096220016479492 seconds
API model: 0.1992943286895752 seconds


⏱ Batch Performance (Code Cell)

In [ ]:
print("⏱ All Chunks Embedding Time")
print("Open-source model:", open_source_time_all, "seconds")
print("API model:", api_time_all, "seconds")


⏱ All Chunks Embedding Time
Open-source model: 0.11094522476196289 seconds
API model: 0.5804600715637207 seconds


# Final Observation

This experiment compared an open-source embedding model and an API-based embedding model using the same set of clearly defined text chunks and queries. The following observations were made:

---

## Semantic Retrieval Accuracy
Both models successfully identified the **“Refund Processing”** chunk as the most relevant result for the original query and its paraphrased version. This confirms that both embedding models are capable of capturing semantic meaning beyond exact keyword matching.

---

## Robustness to Paraphrasing
When the query was paraphrased, the top-ranked chunk remained unchanged for both models. This demonstrates that modern embedding models are **robust to variations in wording** and can handle natural language paraphrasing effectively.

---

## Embedding Dimensions
The open-source model produced **lower-dimensional embeddings**, while the API-based model generated **higher-dimensional embeddings**. Higher-dimensional embeddings generally capture richer semantic information but require more storage and computation.

---

## ⏱ Performance and Latency
The open-source model showed **lower latency**, especially when embedding a single chunk, since it runs locally without network overhead.  
The API-based model introduced **additional latency** due to network calls but offers the benefit of managed infrastructure and consistent performance at scale.

---

## Trade-off Summary
- **Open-source model**: Faster, cost-effective, suitable for local or offline applications.
- **API-based model**: More expressive embeddings, better suited for large-scale, production-grade RAG systems.

---

## Conclusion
The choice of embedding model should be guided by application requirements. For lightweight, low-latency use cases, open-source models are sufficient. For enterprise-grade retrieval systems requiring higher accuracy and scalability, API-based embedding models are a better choice.


# Key Takeaways

- Embedding models convert text into numerical vectors that capture semantic meaning.
- Both open-source and API-based embedding models can accurately retrieve relevant information.
- Modern embedding models handle paraphrased queries effectively, not just exact keywords.
- Embedding dimension affects the richness of semantic representation and computational cost.
- Latency and scalability are important factors when selecting an embedding model for RAG systems.
- The choice of embedding model depends on application requirements such as speed, cost, accuracy, and deployment scale.
